# Argument graph alignment and corrected spectral partitioning

This notebook is a reproducible replacement for **1.7_graph_construction_node_attribute.ipynb**. It uses only `paragraph_dataset_mini`, builds one attributed argument graph per document, computes a global cross-document similarity matrix `H`, and restores argument text from a disjoint clustering.

The corrected pipeline addresses the original notebook's main mathematical and implementation problems:

- documents are loaded before any examples are constructed, so **Run All** follows the saved cell order;
- components and relations are parsed object by object, which keeps fields aligned;
- duplicate relations are preserved in a `MultiDiGraph`;
- cross-graph propagation uses confidence-weighted SUPPORT/ATTACK and incoming/outgoing transition channels;
- the IsoRank update has no per-iteration maximum rescaling and handles isolated nodes explicitly;
- the dense similarity matrix is sparsified before partitioning;
- PageRank, sweep ordering, and conductance all use the same edge weights;
- conductance is evaluated against the full sparse graph, rather than only inside a candidate star;
- assigned nodes are removed from later stars, producing a true partition;
- each cluster is constrained to at most one node from each source graph;
- executable invariants verify symmetry, stochastic walks, convergence, disjointness, coverage, and text restoration.

In [9]:
from __future__ import annotations

import json
import logging
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from scipy import sparse
from transformers import AutoModel, AutoTokenizer

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("argument-graph-pipeline")

RANDOM_SEED = 7
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMBEDDING_BATCH_SIZE = 64

# Cross-graph alignment configuration.
ISORANK_ALPHA = 0.05
ISORANK_TOL = 1e-5
ISORANK_MAX_ITER = 500

# Similarity graph and local spectral partition configuration.
K_NEIGHBORS = 12
MIN_SIMILARITY = 0.55
PPR_RESTART = 0.15
PPR_TOL = 1e-10
PPR_MAX_ITER = 500
STAR_RADIUS = 2
MAX_STAR_NODES = 128
MAX_CONDUCTANCE = 0.80
ONE_NODE_PER_GRAPH = True

logger.info("PyTorch device: %s", DEVICE)

2026-09-24 14:17:07,856 | INFO | PyTorch device: cuda


In [10]:
paragraph_dataset_mini = [
    "116062204", "116075685", "116080900", "116084352", "116084886",
    "116099338", "116102314", "116105977", "116106432", "116109598",
    "116110555", "116116081", "116118921", "116120654", "116121714",
    "116125191", "116126076", "116127831", "116141779", "116142769",
    "116149190", "116150283", "116167789", "116168634", "116170057",
    "116170822", "116192026", "116194637", "116196887", "116197578",
    "116199866", "116204676", "116213690", "116214232", "116220471",
    "116221948", "116226431", "116229939", "116232414", "116233794",
    "116245112", "116246991", "116248468", "116252607", "116253514",
    "116256069", "116257159", "116259036", "116260246", "116261015",
    "116261263", "116270112", "116277632", "116278949", "116283678",
    "116285648", "116289180", "116308208", "116310154", "116320626",
    "116321603", "116336690", "116339406", "116344031", "116350541",
    "116350825", "116358099", "116359089", "116376162", "116377931",
    "116385451", "116398308", "116410200", "116410462", "116414218",
    "116432495", "116434259", "116442566", "116462199", "116588653",
    "116681882", "116764649", "116780853", "116790074", "116794466",
    "116972204", "117069850", "117125134", "117130242", "117230819",
    "117251852", "117420480", "117440182", "117456818", "117460145",
    "117464049", "117489745", "117516936", "117556435",
]

assert paragraph_dataset_mini
assert len(paragraph_dataset_mini) == len(set(paragraph_dataset_mini))
logger.info("Configured %d documents", len(paragraph_dataset_mini))

2026-09-24 14:17:08,378 | INFO | Configured 99 documents


In [11]:
ENCODER_PATH = Path("bge_model/bge-small-uk-law-v1_1/final/encoder")
TOKENIZER_PATH = Path("bge_model/bge-small-uk-law-v1_1/final/tokenizer")
DATA_PATH = Path("document_data_parsing")


def locate_project_resource(relative_path: Path) -> Path:
    """Find a notebook resource whether the kernel starts in the project root or notebook folder."""
    search_roots = (Path.cwd(), *Path.cwd().parents)
    candidates = []
    for root in search_roots:
        candidates.extend(
            [
                root / relative_path,
                root / "src" / "legal_data_doc_parts" / "paragraph_analysis" / relative_path,
                root / "src" / "legal_data_doc_parts" / relative_path,
            ]
        )

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    checked = "; ".join(str(path) for path in candidates)
    raise FileNotFoundError(
        f"Could not locate {relative_path}. Checked: {checked}"
    )


DATA_DIR = locate_project_resource(DATA_PATH)
ENCODER_DIR = locate_project_resource(ENCODER_PATH)
TOKENIZER_DIR = locate_project_resource(TOKENIZER_PATH)

logger.info("Data directory: %s", DATA_DIR)
logger.info("Encoder directory: %s", ENCODER_DIR)

2026-09-24 14:17:10,062 | INFO | Data directory: /home/vegura/Projects/study/legal-data-doc-parts/src/legal_data_doc_parts/paragraph_analysis/document_data_parsing
2026-09-24 14:17:10,066 | INFO | Encoder directory: /home/vegura/Projects/study/legal-data-doc-parts/src/legal_data_doc_parts/bge_model/bge-small-uk-law-v1_1/final/encoder


## Load and validate the source structures

Each component and relation is read as one JSON object. This avoids the positional misalignment that can occur when several independent JSONPath queries skip different missing fields. Invalid references, duplicate component IDs, non-finite confidences, and unsupported data shapes fail early.

In [12]:
def read_documents(document_ids: list[str]) -> dict[str, dict]:
    documents = {}
    for document_id in document_ids:
        path = DATA_DIR / f"{document_id}.json"
        if not path.is_file():
            raise FileNotFoundError(f"Missing structure for document {document_id}: {path}")
        with path.open("r", encoding="utf-8") as stream:
            documents[document_id] = json.load(stream)
    return documents


def parse_documents(
    documents: dict[str, dict],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    component_records = []
    relation_records = []

    for graph_index, document_id in enumerate(paragraph_dataset_mini):
        document = documents[document_id]
        result = document.get("result")
        if not isinstance(result, dict):
            raise ValueError(f"Document {document_id} has no object-valued 'result'")

        components = result.get("components")
        relations = result.get("relations", [])
        if not isinstance(components, list) or not components:
            raise ValueError(f"Document {document_id} has no components")
        if not isinstance(relations, list):
            raise ValueError(f"Document {document_id} relations must be a list")

        component_ids = []
        for local_node_index, component in enumerate(components):
            if not isinstance(component, dict):
                raise TypeError(
                    f"Document {document_id} component {local_node_index} must be an object"
                )

            argument_id = str(component.get("id", "")).strip()
            argument_type = str(component.get("type", "")).strip().upper()
            source_span = component.get("source_span") or {}
            text = source_span.get("text")
            if not argument_id or not argument_type or not isinstance(text, str) or not text.strip():
                raise ValueError(
                    f"Document {document_id} component {local_node_index} "
                    "requires non-empty id, type, and source_span.text"
                )

            component_ids.append(argument_id)
            component_records.append(
                {
                    "global_node_index": len(component_records),
                    "graph_index": graph_index,
                    "local_node_index": local_node_index,
                    "document_id": document_id,
                    "argument_id": argument_id,
                    "node_id": f"{document_id}.{argument_id}",
                    "argument_type": argument_type,
                    "argumentative_role": component.get("argumentative_role"),
                    "section": component.get("section"),
                    "text": text.strip(),
                }
            )

        if len(component_ids) != len(set(component_ids)):
            raise ValueError(f"Document {document_id} contains duplicate component IDs")
        component_id_set = set(component_ids)

        for relation_index, relation in enumerate(relations):
            if not isinstance(relation, dict):
                raise TypeError(
                    f"Document {document_id} relation {relation_index} must be an object"
                )

            source = str(relation.get("source", "")).strip()
            target = str(relation.get("target", "")).strip()
            relation_type = str(relation.get("relation", "")).strip().upper()
            relation_id = str(relation.get("id", relation_index))
            try:
                confidence = float(relation.get("confidence", 1.0))
            except (TypeError, ValueError) as error:
                raise ValueError(
                    f"Document {document_id} relation {relation_id} has invalid confidence"
                ) from error

            if source not in component_id_set or target not in component_id_set:
                raise ValueError(
                    f"Document {document_id} relation {relation_id} references "
                    f"unknown endpoint {source!r} -> {target!r}"
                )
            if not relation_type:
                raise ValueError(
                    f"Document {document_id} relation {relation_id} has no relation type"
                )
            if not np.isfinite(confidence) or not 0.0 <= confidence <= 1.0:
                raise ValueError(
                    f"Document {document_id} relation {relation_id} confidence "
                    f"must be finite and in [0, 1], got {confidence}"
                )

            relation_records.append(
                {
                    "graph_index": graph_index,
                    "relation_index": relation_index,
                    "document_id": document_id,
                    "relation_id": relation_id,
                    "source": source,
                    "target": target,
                    "relation": relation_type,
                    "confidence": confidence,
                    "justification": relation.get("justification"),
                }
            )

    arguments = pd.DataFrame.from_records(component_records)
    relations = pd.DataFrame.from_records(relation_records)
    return arguments, relations


doc_args = read_documents(paragraph_dataset_mini)
arguments_df, relations_df = parse_documents(doc_args)

assert list(doc_args) == paragraph_dataset_mini
assert arguments_df["global_node_index"].tolist() == list(range(len(arguments_df)))
assert not arguments_df[["document_id", "argument_id", "argument_type", "text"]].isna().any().any()

dataset_summary = pd.Series(
    {
        "documents": len(doc_args),
        "arguments": len(arguments_df),
        "relations": len(relations_df),
        "premises": int((arguments_df["argument_type"] == "PREMISE").sum()),
        "claims": int((arguments_df["argument_type"] == "CLAIM").sum()),
        "duplicate_source_target_pairs": int(
            relations_df.duplicated(["document_id", "source", "target"]).sum()
        ),
    },
    name="count",
)
display(dataset_summary.to_frame())

,count
documents,99
arguments,3941
relations,2928
premises,2710
claims,1231
duplicate_source_target_pairs,15


## Batch node embeddings

Argument text and section text are encoded once, in dataset order, on the selected PyTorch device. The same normalized tensors are reused for every graph pair. This removes thousands of single-text encoder calls and guarantees that rows in the embedding matrices match rows and columns in `H`.

In [13]:
tokenizer = AutoTokenizer.from_pretrained(
    str(TOKENIZER_DIR),
    local_files_only=True,
)
argument_encoder = AutoModel.from_pretrained(
    str(ENCODER_DIR),
    local_files_only=True,
    add_pooling_layer=False,
).to(DEVICE)
argument_encoder.eval()


@torch.inference_mode()
def encode_texts(texts: list[str]) -> torch.Tensor:
    batches = []
    for start in range(0, len(texts), EMBEDDING_BATCH_SIZE):
        batch = texts[start : start + EMBEDDING_BATCH_SIZE]
        tokens = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )
        tokens = {
            name: value.to(DEVICE)
            for name, value in tokens.items()
        }
        output = argument_encoder(**tokens)
        token_embeddings = output.last_hidden_state
        attention_mask = tokens["attention_mask"].unsqueeze(-1).to(
            token_embeddings.dtype
        )
        pooled = (
            (token_embeddings * attention_mask).sum(dim=1)
            / attention_mask.sum(dim=1).clamp_min(1.0)
        )
        batches.append(F.normalize(pooled, p=2, dim=1))

    embeddings = torch.cat(batches, dim=0).to(
        device=DEVICE,
        dtype=torch.float32,
    )
    if embeddings.ndim != 2 or embeddings.shape[0] != len(texts):
        raise ValueError(
            f"Encoder returned {tuple(embeddings.shape)} for {len(texts)} texts"
        )
    if not torch.isfinite(embeddings).all():
        raise ValueError("Encoder returned non-finite values")
    return embeddings


text_embeddings = encode_texts(arguments_df["text"].tolist())

section_texts = arguments_df["section"].fillna("").astype(str).str.strip().tolist()
section_present = torch.tensor(
    [bool(section) for section in section_texts],
    dtype=torch.bool,
    device=DEVICE,
)
section_embeddings = encode_texts(
    [section if section else " " for section in section_texts]
)
section_embeddings = section_embeddings * section_present[:, None]

assert text_embeddings.shape == section_embeddings.shape
logger.info(
    "Encoded %d arguments into %d dimensions",
    text_embeddings.shape[0],
    text_embeddings.shape[1],
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

2026-09-24 14:17:25,254 | INFO | Encoded 3941 arguments into 384 dimensions


## Attributed document graphs and relation-aware transitions

Each document becomes a `MultiDiGraph`, so two annotated relations with the same source and target remain distinct. For structural propagation, every edge contributes its confidence to a channel defined by relation type and direction. All channel rows are normalized jointly. A node with no incident relation receives probability one in an isolated self channel, so every combined transition is stochastic.

In [14]:
argument_types = sorted(arguments_df["argument_type"].unique())
type_to_code = {value: index for index, value in enumerate(argument_types)}

role_values = (
    arguments_df["argumentative_role"]
    .fillna("")
    .astype(str)
    .str.strip()
)
roles = sorted(value for value in role_values.unique() if value)
role_to_code = {value: index for index, value in enumerate(roles)}

relation_types = sorted(relations_df["relation"].unique())
channel_names = tuple(
    name
    for relation_type in relation_types
    for name in (f"{relation_type}_out", f"{relation_type}_in")
) + ("isolated",)


def build_transition_channels(graph: nx.MultiDiGraph) -> dict[str, torch.Tensor]:
    nodes = list(graph.nodes)
    positions = {node: index for index, node in enumerate(nodes)}
    matrices = {
        name: np.zeros((len(nodes), len(nodes)), dtype=np.float32)
        for name in channel_names
    }

    for source, target, attributes in graph.edges(data=True):
        relation_type = attributes["relation"]
        confidence = float(attributes["confidence"])
        source_index = positions[source]
        target_index = positions[target]
        matrices[f"{relation_type}_out"][source_index, target_index] += confidence
        matrices[f"{relation_type}_in"][target_index, source_index] += confidence

    total_degree = sum(
        matrix.sum(axis=1)
        for name, matrix in matrices.items()
        if name != "isolated"
    )
    non_isolated = total_degree > 0
    for name in channel_names:
        if name == "isolated":
            matrices[name][~non_isolated, ~non_isolated] = 1.0
        else:
            matrices[name][non_isolated] /= total_degree[non_isolated, None]

    transitions = {
        name: torch.as_tensor(matrix, dtype=torch.float32, device=DEVICE)
        for name, matrix in matrices.items()
    }
    combined_row_sums = sum(matrix.sum(dim=1) for matrix in transitions.values())
    if not torch.allclose(
        combined_row_sums,
        torch.ones_like(combined_row_sums),
        atol=1e-6,
    ):
        raise AssertionError("Relation-channel transition rows must sum to one")
    return transitions


cases: list[nx.MultiDiGraph] = []
case_data: list[dict] = []
graph_sizes = []

for graph_index, document_id in enumerate(paragraph_dataset_mini):
    node_rows = (
        arguments_df.loc[arguments_df["graph_index"] == graph_index]
        .sort_values("local_node_index")
    )
    edge_rows = (
        relations_df.loc[relations_df["graph_index"] == graph_index]
        .sort_values("relation_index")
    )

    graph = nx.MultiDiGraph(document_id=document_id)
    for row in node_rows.itertuples(index=False):
        embedding_index = int(row.global_node_index)
        graph.add_node(
            row.node_id,
            argument_id=row.argument_id,
            arg_type=row.argument_type,
            argumentative_role=row.argumentative_role,
            section=row.section,
            text=row.text,
            embedding=text_embeddings[embedding_index].detach().cpu().numpy(),
            section_embedding=section_embeddings[embedding_index].detach().cpu().numpy(),
        )

    for row in edge_rows.itertuples(index=False):
        graph.add_edge(
            f"{document_id}.{row.source}",
            f"{document_id}.{row.target}",
            key=int(row.relation_index),
            relation_id=row.relation_id,
            relation=row.relation,
            confidence=float(row.confidence),
            justification=row.justification,
        )

    if graph.number_of_nodes() != len(node_rows):
        raise AssertionError(f"Node count mismatch for document {document_id}")
    if graph.number_of_edges() != len(edge_rows):
        raise AssertionError(f"Relation count mismatch for document {document_id}")

    global_indices = node_rows["global_node_index"].to_numpy(
        dtype=np.int64,
        copy=True,
    )
    expected_indices = np.arange(
        global_indices[0],
        global_indices[0] + len(global_indices),
        dtype=np.int64,
    )
    if not np.array_equal(global_indices, expected_indices):
        raise AssertionError(f"Nodes for document {document_id} are not contiguous")

    role_codes = torch.tensor(
        [role_to_code.get(value, -1) for value in role_values.iloc[global_indices]],
        dtype=torch.long,
        device=DEVICE,
    )
    type_codes = torch.tensor(
        [type_to_code[value] for value in node_rows["argument_type"]],
        dtype=torch.long,
        device=DEVICE,
    )
    index_tensor = torch.as_tensor(global_indices, dtype=torch.long, device=DEVICE)

    cases.append(graph)
    graph_sizes.append(graph.number_of_nodes())
    case_data.append(
        {
            "document_id": document_id,
            "graph": graph,
            "text_embeddings": text_embeddings[index_tensor],
            "section_embeddings": section_embeddings[index_tensor],
            "section_present": section_present[index_tensor],
            "type_codes": type_codes,
            "role_codes": role_codes,
            "transitions": build_transition_channels(graph),
        }
    )

offsets = np.concatenate(([0], np.cumsum(graph_sizes))).astype(np.int64)
assert offsets[-1] == len(arguments_df)
assert len(cases) == len(paragraph_dataset_mini)

graph_summary = pd.DataFrame(
    {
        "document_id": paragraph_dataset_mini,
        "nodes": graph_sizes,
        "relations": [graph.number_of_edges() for graph in cases],
        "isolated_nodes": [
            sum(graph.degree(node) == 0 for node in graph.nodes)
            for graph in cases
        ],
    }
)
display(graph_summary.describe().round(2))

,nodes,relations,isolated_nodes
count,99.00,99.00,99.00
mean,39.81,29.58,10.12
std,21.55,13.55,11.70
min,1.00,0.00,0.00
25%,28.00,22.00,3.00
50%,37.00,30.00,7.00
75%,47.50,37.00,11.50
max,132.00,66.00,67.00


## Corrected cross-graph similarity

For graph pair $a,b$, the prior $E$ combines cosine similarity of argument text, cosine similarity of section text when both sections exist, and an argumentative-role match. Pairs with different argument types are masked out.

Let $P_{a,r}$ and $P_{b,r}$ be the jointly normalized transition matrices for a relation-and-direction channel $r$. The iteration is

$$
S^{(t+1)}=(1-\alpha)E+\alpha\left(\sum_r P_{a,r}S^{(t)}P_{b,r}^{\mathsf T}\right).
$$

Because all channel weights are nonnegative, their joint row sums equal one, and $0\leq\alpha<1$, this is a contraction on bounded scores. Unlike the original update, it does not rescale every structural term by its current maximum. SUPPORT and ATTACK are matched only to the same relation and direction.

In [15]:
@torch.inference_mode()
def node_attribute_prior(first: dict, second: dict) -> tuple[torch.Tensor, torch.Tensor]:
    type_mask = first["type_codes"][:, None] == second["type_codes"][None, :]

    text_similarity = (
        first["text_embeddings"] @ second["text_embeddings"].T
    ).clamp(0.0, 1.0)
    section_similarity = (
        first["section_embeddings"] @ second["section_embeddings"].T
    ).clamp(0.0, 1.0)

    section_available = (
        first["section_present"][:, None] & second["section_present"][None, :]
    )
    role_available = (
        (first["role_codes"][:, None] >= 0)
        & (second["role_codes"][None, :] >= 0)
    )
    role_match = (
        first["role_codes"][:, None] == second["role_codes"][None, :]
    ) & role_available

    numerator = (
        0.9 * text_similarity
        + 0.20 * section_similarity * section_available
        + 0.05 * role_match
    )
    denominator = (
        0.8
        + 0.20 * section_available.to(torch.float32)
        + 0.05 * role_available.to(torch.float32)
    )
    prior = (numerator / denominator) * type_mask
    return prior.to(torch.float32), type_mask


@torch.inference_mode()
def cross_graph_similarity(
    first: dict,
    second: dict,
    *,
    alpha: float = ISORANK_ALPHA,
    tol: float = ISORANK_TOL,
    max_iter: int = ISORANK_MAX_ITER,
) -> tuple[torch.Tensor, int, float]:
    if not 0.0 <= alpha < 1.0:
        raise ValueError("alpha must satisfy 0 <= alpha < 1")
    if tol <= 0 or max_iter <= 0:
        raise ValueError("tol and max_iter must be positive")

    prior, type_mask = node_attribute_prior(first, second)
    score = prior.clone()
    delta = float("inf")

    for iteration in range(1, max_iter + 1):
        propagated = torch.zeros_like(score)
        for channel in channel_names:
            propagated.add_(
                first["transitions"][channel]
                @ score
                @ second["transitions"][channel].T
            )

        new_score = ((1.0 - alpha) * prior + alpha * propagated) * type_mask
        delta = torch.max(torch.abs(new_score - score)).item()
        score = new_score
        if delta <= tol:
            break

    if not torch.isfinite(score).all():
        raise FloatingPointError("Cross-graph similarity produced non-finite values")
    if torch.any(score < -1e-7) or torch.any(score > 1.0 + 1e-6):
        raise FloatingPointError("Cross-graph scores left the expected [0, 1] range")

    return score.clamp_(0.0, 1.0), iteration, delta


# Small deterministic sanity check before all document pairs are processed.
test_score, test_iterations, test_delta = cross_graph_similarity(
    case_data[0],
    case_data[1],
)
assert test_score.shape == (graph_sizes[0], graph_sizes[1])
assert test_delta <= ISORANK_TOL
logger.info(
    "Pairwise similarity sanity check: shape=%s, iterations=%d, delta=%.3g",
    tuple(test_score.shape),
    test_iterations,
    test_delta,
)

2026-09-24 14:17:31,109 | INFO | Pairwise similarity sanity check: shape=(87, 132), iterations=4, delta=1.49e-06


## Global matrix $H$

Rows and columns follow `arguments_df.global_node_index`. Only cross-document blocks are populated. The lower block is the exact transpose of the upper block, so $H$ is symmetric; within-document blocks and the diagonal remain zero.

In [16]:
@torch.inference_mode()
def build_global_similarity_matrix(
    attributed_cases: list[dict],
) -> tuple[np.ndarray, pd.DataFrame]:
    total_nodes = int(offsets[-1])
    matrix = torch.zeros(
        (total_nodes, total_nodes),
        dtype=torch.float32,
        device=DEVICE,
    )
    diagnostics = []
    pair_total = len(attributed_cases) * (len(attributed_cases) - 1) // 2
    pair_number = 0

    for first_index in range(len(attributed_cases)):
        first_slice = slice(offsets[first_index], offsets[first_index + 1])
        for second_index in range(first_index + 1, len(attributed_cases)):
            second_slice = slice(offsets[second_index], offsets[second_index + 1])
            score, iterations, delta = cross_graph_similarity(
                attributed_cases[first_index],
                attributed_cases[second_index],
            )
            matrix[first_slice, second_slice] = score
            matrix[second_slice, first_slice] = score.T

            diagnostics.append(
                {
                    "graph_1": first_index,
                    "graph_2": second_index,
                    "iterations": iterations,
                    "final_delta": delta,
                    "converged": delta <= ISORANK_TOL,
                    "nonzero_scores": int(torch.count_nonzero(score).item()),
                }
            )
            pair_number += 1
            if pair_number % 500 == 0 or pair_number == pair_total:
                logger.info("Aligned %d/%d graph pairs", pair_number, pair_total)

    return matrix.cpu().numpy(), pd.DataFrame.from_records(diagnostics)


H, alignment_diagnostics = build_global_similarity_matrix(case_data)
logger.info("Global similarity matrix shape: %s", H.shape)

FloatingPointError: Cross-graph scores left the expected [0, 1] range

In [ ]:
expected_shape = (len(arguments_df), len(arguments_df))
if H.shape != expected_shape:
    raise AssertionError(f"Expected H shape {expected_shape}, got {H.shape}")
if not np.isfinite(H).all():
    raise AssertionError("H contains NaN or infinity")
if H.min() < -1e-7 or H.max() > 1.0 + 1e-6:
    raise AssertionError("H contains scores outside [0, 1]")
if not np.allclose(H, H.T, atol=1e-6):
    raise AssertionError("H is not symmetric")
if not np.allclose(np.diag(H), 0.0, atol=1e-7):
    raise AssertionError("H diagonal must be zero")
if np.count_nonzero(H) == 0:
    raise AssertionError("H contains only zeros")
if not alignment_diagnostics["converged"].all():
    failed = alignment_diagnostics.loc[~alignment_diagnostics["converged"]]
    raise AssertionError(f"{len(failed)} graph pairs did not converge")

for graph_index in range(len(cases)):
    block = H[
        offsets[graph_index] : offsets[graph_index + 1],
        offsets[graph_index] : offsets[graph_index + 1],
    ]
    if np.count_nonzero(block):
        raise AssertionError(f"Within-document block {graph_index} must be zero")

row_maxima = H.max(axis=1)
if np.any(row_maxima <= 0):
    zero_rows = np.flatnonzero(row_maxima <= 0)
    raise AssertionError(f"H has {len(zero_rows)} rows without a cross-document match")

h_summary = pd.Series(
    {
        "nodes": H.shape[0],
        "nonzero_entries": int(np.count_nonzero(H)),
        "density": float(np.count_nonzero(H) / H.size),
        "minimum": float(H.min()),
        "maximum": float(H.max()),
        "mean_positive": float(H[H > 0].mean()),
        "median_row_maximum": float(np.median(row_maxima)),
        "max_alignment_iterations": int(alignment_diagnostics["iterations"].max()),
        "max_final_delta": float(alignment_diagnostics["final_delta"].max()),
    },
    name="value",
)
display(h_summary.to_frame())

## Sparse affinity graph for spectral partitioning

Raw cosine-based similarities are dense: almost every same-type argument has a positive score. Conductance on that complete weighted graph is therefore close to one for nearly every small set.

The partitioning graph first keeps an edge only when both endpoints select it among their $k$ strongest cross-document matches and its weight passes the absolute floor. To prevent the strict mutual rule from turning otherwise valid arguments into forced singletons, each node left isolated receives only its single strongest qualifying edge. The graph stays symmetric and retains the original similarity as every edge weight.

In [ ]:
def mutual_knn_graph(
    similarity: np.ndarray,
    *,
    neighbors: int,
    minimum_similarity: float,
) -> tuple[sparse.csr_matrix, int, int]:
    if similarity.ndim != 2 or similarity.shape[0] != similarity.shape[1]:
        raise ValueError("similarity must be a square matrix")
    if neighbors <= 0:
        raise ValueError("neighbors must be positive")

    node_count = similarity.shape[0]
    k = min(neighbors, node_count - 1)
    tensor = torch.from_numpy(similarity)
    values, columns = torch.topk(tensor, k=k, dim=1)

    rows = torch.arange(node_count)[:, None].expand_as(columns)
    keep = (
        (values >= minimum_similarity)
        & (columns != rows)
        & torch.isfinite(values)
    )
    directed = sparse.coo_matrix(
        (
            values[keep].cpu().numpy(),
            (
                rows[keep].cpu().numpy(),
                columns[keep].cpu().numpy(),
            ),
        ),
        shape=(node_count, node_count),
        dtype=np.float32,
    ).tocsr()

    affinity = directed.minimum(directed.T).tocsr()
    affinity.setdiag(0.0)
    affinity.eliminate_zeros()
    affinity = ((affinity + affinity.T) * 0.5).tocsr()

    degree_before_repair = np.asarray(affinity.sum(axis=1)).ravel()
    isolated_before_repair = np.flatnonzero(degree_before_repair == 0)
    edge_count_before_repair = affinity.nnz // 2

    repaired = affinity.tolil(copy=True)
    for node in isolated_before_repair:
        target = int(columns[node, 0])
        weight = float(values[node, 0])
        if target != node and weight >= minimum_similarity:
            repaired[node, target] = weight
            repaired[target, node] = weight

    affinity = repaired.tocsr()
    affinity.setdiag(0.0)
    affinity.eliminate_zeros()
    affinity = ((affinity + affinity.T) * 0.5).tocsr()
    affinity.sort_indices()

    fallback_edges = affinity.nnz // 2 - edge_count_before_repair
    return affinity, len(isolated_before_repair), fallback_edges


H_graph, isolated_before_repair, fallback_edges = mutual_knn_graph(
    H,
    neighbors=K_NEIGHBORS,
    minimum_similarity=MIN_SIMILARITY,
)
weighted_degree = np.asarray(H_graph.sum(axis=1)).ravel()
isolated_indices = np.flatnonzero(weighted_degree == 0)
node_graph_index = arguments_df["graph_index"].to_numpy(
    dtype=np.int64,
    copy=True,
)
edge_rows, edge_columns = H_graph.nonzero()
same_graph_edges = int(
    np.count_nonzero(
        node_graph_index[edge_rows] == node_graph_index[edge_columns]
    )
)

if H_graph.nnz == 0:
    raise AssertionError("Similarity sparsification removed every edge")
if (H_graph - H_graph.T).nnz:
    raise AssertionError("Sparse affinity graph is not exactly symmetric")
if same_graph_edges:
    raise AssertionError(
        f"Sparse affinity graph contains {same_graph_edges} within-graph edges"
    )
if np.any(H_graph.data <= 0) or not np.isfinite(H_graph.data).all():
    raise AssertionError("Sparse affinity weights must be finite and positive")

sparse_summary = pd.Series(
    {
        "undirected_edges": H_graph.nnz // 2,
        "mutual_knn_isolates_before_repair": isolated_before_repair,
        "fallback_edges_added": fallback_edges,
        "isolated_nodes_after_repair": len(isolated_indices),
        "within_graph_directed_edges": same_graph_edges,
        "mean_degree": H_graph.nnz / H_graph.shape[0],
        "minimum_edge_weight": float(H_graph.data.min()),
        "maximum_edge_weight": float(H_graph.data.max()),
    },
    name="value",
)
display(sparse_summary.to_frame())

## Weighted local spectral partitioning

For a candidate neighborhood, the algorithm creates the lazy walk $W=(I+D^{-1}A)/2$, including a self transition for a zero-degree row. Personalized PageRank is solved as

$$
p=\gamma e_s+(1-\gamma)W^{\mathsf T}p.
$$

Nodes are ordered by $p(v)/d_H(v)$, where $d_H$ is the degree in the full sparse affinity graph. Every prefix containing the seed is evaluated with full-graph weighted conductance

$$
\phi(S)=\frac{w(S,\bar S)}
{\min(\operatorname{vol}(S),\operatorname{vol}(\bar S))}.
$$

The full graph in this formula fixes the original notebook's induced-star boundary error. Candidate neighborhoods are bounded two-hop regions for efficiency. Once a cluster is accepted, all its nodes are removed from later candidate neighborhoods, so the result is a disjoint partition.

With `ONE_NODE_PER_GRAPH=True`, the sweep keeps only the highest-ranked candidate from each source graph and reserves the seed's source graph for the seed. The PageRank walk may still use all candidate nodes to estimate spectral importance, but an accepted cluster cannot contain two arguments from the same document.

In [ ]:
def lazy_random_walk(
    adjacency: sparse.csr_matrix,
) -> tuple[sparse.csr_matrix, np.ndarray, float]:
    adjacency = adjacency.tocsr().astype(np.float64)
    if adjacency.shape[0] != adjacency.shape[1]:
        raise ValueError("adjacency must be square")
    if (adjacency - adjacency.T).nnz:
        raise ValueError("adjacency must be symmetric")

    degree = np.asarray(adjacency.sum(axis=1)).ravel()
    inverse_degree = np.zeros_like(degree)
    positive = degree > 0
    inverse_degree[positive] = 1.0 / degree[positive]

    transition = sparse.diags(inverse_degree) @ adjacency
    if np.any(~positive):
        transition = transition + sparse.diags((~positive).astype(np.float64))

    identity = sparse.identity(adjacency.shape[0], format="csr", dtype=np.float64)
    walk = ((identity + transition) * 0.5).tocsr()
    row_sums = np.asarray(walk.sum(axis=1)).ravel()
    row_error = float(np.max(np.abs(row_sums - 1.0))) if len(row_sums) else 0.0
    if row_error > 1e-10:
        raise AssertionError(f"Lazy walk is not row-stochastic; max error={row_error}")
    return walk, degree, row_error


def personalized_pagerank(
    walk: sparse.csr_matrix,
    *,
    seed_index: int,
    restart: float = PPR_RESTART,
    tol: float = PPR_TOL,
    max_iter: int = PPR_MAX_ITER,
) -> tuple[np.ndarray, int, float]:
    if not 0.0 < restart <= 1.0:
        raise ValueError("restart must satisfy 0 < restart <= 1")
    node_count = walk.shape[0]
    if not 0 <= seed_index < node_count:
        raise IndexError("seed_index is outside the walk")

    seed = np.zeros(node_count, dtype=np.float64)
    seed[seed_index] = 1.0
    probability = seed.copy()
    delta = float("inf")

    for iteration in range(1, max_iter + 1):
        new_probability = (
            restart * seed
            + (1.0 - restart) * (walk.T @ probability)
        )
        delta = float(np.linalg.norm(new_probability - probability, ord=1))
        probability = np.asarray(new_probability).ravel()
        if delta <= tol:
            break

    if delta > tol:
        raise RuntimeError(
            f"Personalized PageRank did not converge in {max_iter} iterations; "
            f"delta={delta}"
        )
    if not np.isclose(probability.sum(), 1.0, atol=1e-8):
        raise AssertionError(
            f"Personalized PageRank mass is {probability.sum()}, expected one"
        )
    if np.any(probability < -1e-12) or not np.isfinite(probability).all():
        raise FloatingPointError("Personalized PageRank contains invalid values")

    return probability, iteration, delta


def weighted_conductance(
    adjacency: sparse.csr_matrix,
    subset: np.ndarray | list[int],
    *,
    degree: np.ndarray | None = None,
    total_volume: float | None = None,
) -> float:
    nodes = np.unique(np.asarray(subset, dtype=np.int64))
    if nodes.size == 0:
        return float("inf")

    if degree is None:
        degree = np.asarray(adjacency.sum(axis=1)).ravel()
    if total_volume is None:
        total_volume = float(degree.sum())

    volume = float(degree[nodes].sum())
    complement_volume = total_volume - volume
    denominator = min(volume, complement_volume)
    if denominator <= 0:
        return float("inf")

    internal_weight = float(adjacency[nodes][:, nodes].sum())
    boundary_weight = max(volume - internal_weight, 0.0)
    return boundary_weight / denominator


def neighborhood_pool(
    adjacency: sparse.csr_matrix,
    dense_similarity: np.ndarray,
    *,
    center: int,
    allowed: np.ndarray,
    radius: int = STAR_RADIUS,
    max_nodes: int = MAX_STAR_NODES,
) -> np.ndarray:
    if not allowed[center]:
        raise ValueError("center must be unassigned")
    if radius < 1 or max_nodes < 1:
        raise ValueError("radius and max_nodes must be positive")

    selected = [int(center)]
    seen = {int(center)}
    frontier = [int(center)]

    for _ in range(radius):
        discovered = set()
        for node in frontier:
            start, end = adjacency.indptr[node], adjacency.indptr[node + 1]
            for neighbor in adjacency.indices[start:end]:
                neighbor = int(neighbor)
                if allowed[neighbor] and neighbor not in seen:
                    discovered.add(neighbor)

        if not discovered:
            break

        ranked = sorted(
            discovered,
            key=lambda node: (
                -float(dense_similarity[center, node]),
                -float(weighted_degree[node]),
                node,
            ),
        )
        available_slots = max_nodes - len(selected)
        accepted = ranked[:available_slots]
        selected.extend(accepted)
        seen.update(accepted)
        frontier = accepted
        if not frontier or len(selected) >= max_nodes:
            break

    return np.asarray(selected, dtype=np.int64)


def graph_unique_sweep_order(
    order_local: np.ndarray,
    pool: np.ndarray,
    graph_index_by_node: np.ndarray,
    *,
    seed_local_index: int = 0,
) -> np.ndarray:
    """Keep the highest-ranked candidate per graph while preserving the seed."""
    if seed_local_index not in order_local:
        raise AssertionError("Sweep order does not contain its seed")

    seed_graph = int(graph_index_by_node[pool[seed_local_index]])
    selected_order = []
    seen_graphs = set()

    for local_index in order_local:
        local_index = int(local_index)
        graph_index = int(graph_index_by_node[pool[local_index]])

        # Reserve the seed graph for the seed even if another node from that
        # graph has a higher normalized PageRank score.
        if graph_index == seed_graph and local_index != seed_local_index:
            continue
        if graph_index in seen_graphs:
            continue

        selected_order.append(local_index)
        seen_graphs.add(graph_index)

    if seed_local_index not in selected_order:
        raise AssertionError("Graph-constrained sweep lost its seed")

    result = np.asarray(selected_order, dtype=np.int64)
    represented_graphs = graph_index_by_node[pool[result]]
    if len(represented_graphs) != len(np.unique(represented_graphs)):
        raise AssertionError("Graph-constrained sweep contains duplicate graphs")
    return result


def spectral_partition_star(
    adjacency: sparse.csr_matrix,
    dense_similarity: np.ndarray,
    *,
    center: int,
    allowed: np.ndarray,
    maximum_conductance: float = MAX_CONDUCTANCE,
) -> dict:
    pool = neighborhood_pool(
        adjacency,
        dense_similarity,
        center=center,
        allowed=allowed,
    )
    if pool[0] != center or len(pool) != len(np.unique(pool)):
        raise AssertionError("Candidate pool must contain the center exactly once at index zero")

    local_affinity = adjacency[pool][:, pool].tocsr()
    walk, local_degree, row_error = lazy_random_walk(local_affinity)
    pagerank, iterations, delta = personalized_pagerank(
        walk,
        seed_index=0,
    )

    global_degree = np.asarray(adjacency.sum(axis=1)).ravel()
    normalized_score = np.full_like(pagerank, -np.inf)
    positive_degree = global_degree[pool] > 0
    normalized_score[positive_degree] = (
        pagerank[positive_degree] / global_degree[pool][positive_degree]
    )
    order_local = np.argsort(-normalized_score, kind="stable")
    if ONE_NODE_PER_GRAPH:
        order_local = graph_unique_sweep_order(
            order_local,
            pool,
            node_graph_index,
            seed_local_index=0,
        )

    total_volume = float(global_degree.sum())
    best_nodes = None
    best_conductance = float("inf")
    sweep_values = []

    for prefix_size in range(2, len(order_local) + 1):
        prefix_local = order_local[:prefix_size]
        if 0 not in prefix_local:
            continue

        prefix_global = pool[prefix_local]
        conductance = weighted_conductance(
            adjacency,
            prefix_global,
            degree=global_degree,
            total_volume=total_volume,
        )
        sweep_values.append((prefix_size, conductance))
        if conductance < best_conductance:
            best_nodes = prefix_global.copy()
            best_conductance = conductance

    accepted = (
        best_nodes is not None
        and np.isfinite(best_conductance)
        and best_conductance <= maximum_conductance
    )
    cluster_nodes = best_nodes if accepted else np.asarray([center], dtype=np.int64)
    actual_conductance = weighted_conductance(
        adjacency,
        cluster_nodes,
        degree=global_degree,
        total_volume=total_volume,
    )

    return {
        "center": int(center),
        "nodes": np.asarray(cluster_nodes, dtype=np.int64),
        "candidate_nodes": int(len(pool)),
        "candidate_graphs": int(len(np.unique(node_graph_index[pool]))),
        "accepted_sweep": bool(accepted),
        "conductance": float(actual_conductance),
        "best_sweep_conductance": float(best_conductance),
        "pagerank_iterations": int(iterations),
        "pagerank_delta": float(delta),
        "pagerank_mass": float(pagerank.sum()),
        "walk_row_error": float(row_error),
        "sweep_values": sweep_values,
    }


# The graph constraint reserves the seed's graph and keeps one candidate per graph.
_toy_unique_order = graph_unique_sweep_order(
    np.array([1, 2, 0, 3]),
    np.array([0, 1, 2, 3]),
    np.array([0, 0, 1, 2]),
    seed_local_index=0,
)
assert np.array_equal(_toy_unique_order, np.array([2, 0, 3]))

# A two-node weighted graph exercises dangling-safe walk and PPR invariants.
_toy_affinity = sparse.csr_matrix(np.array([[0.0, 0.8], [0.8, 0.0]]))
_toy_walk, _, _toy_row_error = lazy_random_walk(_toy_affinity)
_toy_pagerank, _, _toy_delta = personalized_pagerank(
    _toy_walk,
    seed_index=0,
)
assert _toy_row_error <= 1e-10
assert _toy_delta <= PPR_TOL
assert np.isclose(_toy_pagerank.sum(), 1.0)

In [ ]:
def partition_similarity_graph(
    adjacency: sparse.csr_matrix,
    dense_similarity: np.ndarray,
) -> tuple[list[dict], pd.DataFrame]:
    node_count = adjacency.shape[0]
    degree = np.asarray(adjacency.sum(axis=1)).ravel()
    # Highest-degree seeds are processed first; node index makes ties deterministic.
    priority = np.lexsort((np.arange(node_count), -degree))
    unassigned = np.ones(node_count, dtype=bool)
    clusters = []

    for center in priority:
        center = int(center)
        if not unassigned[center]:
            continue

        result = spectral_partition_star(
            adjacency,
            dense_similarity,
            center=center,
            allowed=unassigned,
        )
        selected = np.unique(result["nodes"])
        if center not in selected:
            raise AssertionError("A spectral cluster lost its seed")
        if not np.all(unassigned[selected]):
            raise AssertionError("A spectral cluster contains an already assigned node")

        result["nodes"] = selected
        result["cluster_id"] = len(clusters)
        clusters.append(result)
        unassigned[selected] = False

    if np.any(unassigned):
        raise AssertionError("Partition left nodes unassigned")

    diagnostics = pd.DataFrame(
        [
            {
                "cluster_id": cluster["cluster_id"],
                "center": cluster["center"],
                "size": len(cluster["nodes"]),
                "candidate_nodes": cluster["candidate_nodes"],
                "candidate_graphs": cluster["candidate_graphs"],
                "accepted_sweep": cluster["accepted_sweep"],
                "conductance": cluster["conductance"],
                "best_sweep_conductance": cluster["best_sweep_conductance"],
                "pagerank_iterations": cluster["pagerank_iterations"],
                "pagerank_delta": cluster["pagerank_delta"],
                "pagerank_mass": cluster["pagerank_mass"],
                "walk_row_error": cluster["walk_row_error"],
            }
            for cluster in clusters
        ]
    )
    return clusters, diagnostics


clusters, partition_diagnostics = partition_similarity_graph(H_graph, H)
cluster_sizes = partition_diagnostics["size"].to_numpy()

logger.info(
    "Partitioned %d nodes into %d clusters (%d multi-node, %d singleton)",
    H.shape[0],
    len(clusters),
    int((cluster_sizes > 1).sum()),
    int((cluster_sizes == 1).sum()),
)

display_diagnostics = partition_diagnostics[
    [
        "size",
        "accepted_sweep",
        "conductance",
        "candidate_nodes",
        "candidate_graphs",
        "pagerank_iterations",
    ]
].copy()
display_diagnostics["conductance"] = display_diagnostics[
    "conductance"
].replace([np.inf, -np.inf], np.nan)
display(display_diagnostics.describe().round(4))

## Partition validation and text restoration

The checks below are part of the pipeline. They prove that every row of $H$ occurs in exactly one cluster, every cluster contains its seed, accepted conductance values match a fresh full-graph calculation, every PageRank vector converged with unit mass, and each global index restores to a non-empty source text.

In [ ]:
all_members = np.concatenate([cluster["nodes"] for cluster in clusters])
expected_members = np.arange(H.shape[0], dtype=np.int64)

if len(all_members) != H.shape[0]:
    raise AssertionError(
        f"Partition has {len(all_members)} memberships for {H.shape[0]} nodes"
    )
if not np.array_equal(np.sort(all_members), expected_members):
    raise AssertionError("Clusters overlap or fail to cover every global node")
if not all(cluster["center"] in cluster["nodes"] for cluster in clusters):
    raise AssertionError("At least one cluster does not contain its center")
if partition_diagnostics["walk_row_error"].max() > 1e-10:
    raise AssertionError("At least one lazy walk is not row-stochastic")
if partition_diagnostics["pagerank_delta"].max() > PPR_TOL:
    raise AssertionError("At least one PageRank solve did not converge")
if not np.allclose(
    partition_diagnostics["pagerank_mass"],
    1.0,
    atol=1e-8,
):
    raise AssertionError("At least one PageRank vector lost probability mass")
if not np.any(cluster_sizes > 1):
    raise AssertionError(
        "No multi-node clusters were produced; inspect kNN and conductance parameters"
    )

multi_node_diagnostics = partition_diagnostics.loc[
    partition_diagnostics["size"] > 1
]
if not multi_node_diagnostics["accepted_sweep"].all():
    raise AssertionError("Every multi-node cluster must come from an accepted sweep")
if (
    multi_node_diagnostics["conductance"] > MAX_CONDUCTANCE + 1e-10
).any():
    raise AssertionError("A multi-node cluster exceeds the conductance threshold")

node_types = arguments_df["argument_type"].to_numpy()
for cluster in clusters:
    if len(np.unique(node_types[cluster["nodes"]])) != 1:
        raise AssertionError(
            f"Cluster {cluster['cluster_id']} mixes argument types"
        )
    cluster_graphs = node_graph_index[cluster["nodes"]]
    if ONE_NODE_PER_GRAPH and len(cluster_graphs) != len(np.unique(cluster_graphs)):
        raise AssertionError(
            f"Cluster {cluster['cluster_id']} contains multiple nodes "
            "from the same source graph"
        )

full_degree = np.asarray(H_graph.sum(axis=1)).ravel()
full_volume = float(full_degree.sum())
for cluster in clusters:
    recalculated = weighted_conductance(
        H_graph,
        cluster["nodes"],
        degree=full_degree,
        total_volume=full_volume,
    )
    if not (
        (np.isinf(recalculated) and np.isinf(cluster["conductance"]))
        or np.isclose(recalculated, cluster["conductance"], atol=1e-10)
    ):
        raise AssertionError(
            f"Conductance mismatch for cluster {cluster['cluster_id']}"
        )

context = arguments_df.set_index("global_node_index", drop=False)
restored_frames = []
for cluster in clusters:
    rows = context.loc[cluster["nodes"]].copy()
    rows.insert(0, "cluster_id", cluster["cluster_id"])
    rows.insert(1, "cluster_size", len(cluster["nodes"]))
    rows.insert(
        2,
        "is_center",
        rows["global_node_index"].eq(cluster["center"]),
    )
    rows.insert(3, "conductance", cluster["conductance"])
    restored_frames.append(rows)

restored_cluster_arguments = pd.concat(restored_frames, ignore_index=True)
restored_cluster_arguments = restored_cluster_arguments.sort_values(
    ["cluster_id", "is_center", "global_node_index"],
    ascending=[True, False, True],
).reset_index(drop=True)

if len(restored_cluster_arguments) != H.shape[0]:
    raise AssertionError("Restored output does not contain exactly one row per node")
if restored_cluster_arguments["global_node_index"].nunique() != H.shape[0]:
    raise AssertionError("Restored output duplicates or omits global node indices")
if (
    restored_cluster_arguments["text"].isna().any()
    or restored_cluster_arguments["text"].str.strip().eq("").any()
):
    raise AssertionError("Restored output contains missing argument text")

cluster_text_context = {
    int(cluster_id): group[
        [
            "global_node_index",
            "document_id",
            "argument_id",
            "argument_type",
            "argumentative_role",
            "section",
            "text",
        ]
    ].to_dict("records")
    for cluster_id, group in restored_cluster_arguments.groupby(
        "cluster_id",
        sort=True,
    )
}

validation_summary = pd.Series(
    {
        "nodes_in_H": H.shape[0],
        "cluster_memberships": len(all_members),
        "unique_cluster_members": len(np.unique(all_members)),
        "clusters": len(clusters),
        "type_homogeneous_clusters": len(clusters),
        "source_graph_unique_clusters": len(clusters),
        "within_graph_affinity_edges": same_graph_edges,
        "multi_node_clusters": int((cluster_sizes > 1).sum()),
        "singletons": int((cluster_sizes == 1).sum()),
        "largest_cluster": int(cluster_sizes.max()),
        "restored_nonempty_texts": int(
            restored_cluster_arguments["text"].str.strip().ne("").sum()
        ),
    },
    name="value",
)
display(validation_summary.to_frame())

In [ ]:
multi_node_cluster_ids = (
    partition_diagnostics.loc[partition_diagnostics["size"] > 1]
    .sort_values(["size", "conductance"], ascending=[False, True])
    ["cluster_id"]
    # .head(5)
    .tolist()
)

example_cluster_arguments = restored_cluster_arguments.loc[
    restored_cluster_arguments["cluster_id"].isin(multi_node_cluster_ids),
    [
        "cluster_id",
        "cluster_size",
        "is_center",
        "conductance",
        "global_node_index",
        "document_id",
        "argument_id",
        "argument_type",
        "argumentative_role",
        "text",
    ],
]
display(example_cluster_arguments.head(100))

# Complete context remains available in cluster_text_context.
example_cluster_id = int(multi_node_cluster_ids[0])
example_cluster_context = pd.DataFrame(
    cluster_text_context[example_cluster_id]
)
display(example_cluster_context.head(20))

### Outputs

- `H`: dense, symmetric global cross-document similarity matrix.
- `H_graph`: symmetric weighted mutual-kNN affinity graph used by the spectral sweep.
- `clusters`: disjoint partition expressed with global row/column indices of `H`.
- `partition_diagnostics`: convergence, candidate size, and conductance for every cluster.
- `restored_cluster_arguments`: one row per argument with cluster ID and restored metadata/text.
- `cluster_text_context`: dictionary from cluster ID to the complete argument context records.

In [ ]:
print(
    partition_diagnostics[
        [
            "cluster_id",
            "center",
            "size",
            "candidate_graphs",
            "accepted_sweep",
            "conductance",
        ]
    ].head(10).to_string(index=False)
)

In [ ]:
example_cluster_arguments[example_cluster_arguments["cluster_id"] == 1114]["text"].tolist()

In [ ]:
example_cluster_arguments[example_cluster_arguments["cluster_id"] == 949]["text"].tolist()

In [ ]:
example_cluster_arguments[example_cluster_arguments["cluster_id"] == 582]["text"].tolist()

In [ ]:
example_cluster_arguments[example_cluster_arguments["cluster_id"] == 147]["text"].tolist()

In [ ]:
example_cluster_arguments["cluster_id"]

In [ ]:
restored_cluster_arguments.to_parquet("clustered_arguments.parquet.gzip")

In [ ]:
# окрема думка
#
#
# достатність допустимість
# - верховний суд практика
# - оцінка відносно судді
#
#
# liga360